# 01_05: Question Answering

**Objective:** Learn to build extractive question answering systems that find answer spans within a given context passage.

**Prerequisites:** Basic Python, familiarity with HuggingFace pipelines (Notebook 00_03).

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|-------------|------------|------|---------|-------------------|-------|
| **Small (CPU)** | distilbert-base-cased-distilled-squad | ~260MB | 4GB RAM | Any CPU | Default for this notebook |
| **Large (GPU)** | deepset/roberta-base-squad2 | ~500MB | 8GB RAM | RTX 4080+ | Handles unanswerable questions |

## Expected Behaviors

### First Time Running
- **Model download**: ~260MB for DistilBERT-QA (1-2 minutes)
- Cached in `~/.cache/huggingface/hub/` for subsequent runs

### What You'll See
- Models extract exact answer spans from context text (not generating new text)
- Confidence scores indicate how certain the model is about its answer
- Longer contexts and more specific questions generally yield better results

### Common Observations
- Extractive QA only returns text that exists in the context — it never generates new words
- Questions about information not in the context get low-confidence (often wrong) answers
- The model finds start and end positions within the tokenized context

## Overview

**Extractive Question Answering** takes two inputs:
1. A **context** passage containing information
2. A **question** about that information

The model outputs the **start and end positions** of the answer within the context. This is fundamentally different from generative QA (like ChatGPT), which creates new text.

### How It Works

```
Context: "The Eiffel Tower is located in Paris, France. It was built in 1889."
Question: "When was the Eiffel Tower built?"

Model predicts:
  Start position → "1889"
  End position   → "1889"
  Answer: "1889" (confidence: 0.97)
```

The most popular benchmark for this task is **SQuAD** (Stanford Question Answering Dataset), which contains 100k+ question-context-answer triples from Wikipedia.

## Setup and Installation

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForQuestionAnswering,
)

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# Version metadata
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Model Selection

In [ ]:
# Option 1: Small Model (CPU-friendly, recommended for beginners)
MODEL_NAME = 'distilbert-base-cased-distilled-squad'  # ~260MB, SQuAD v1

# Option 2: Large Model (GPU-optimized, handles unanswerable questions)
# MODEL_NAME = 'deepset/roberta-base-squad2'           # ~500MB, SQuAD v2

## Method 1: Pipeline API

The pipeline handles all the complexity of tokenization, model inference, and answer span extraction. You provide a question and a context, and it returns the answer with a confidence score.

In [ ]:
# Create QA pipeline
qa_pipeline = pipeline(
    'question-answering',
    model=MODEL_NAME,
    device=device,
)

# Simple example
context = (
    'The Transformer architecture was introduced in 2017 by researchers at Google '
    'in the paper "Attention Is All You Need". The key innovation was the self-attention '
    'mechanism, which allows the model to weigh the importance of different parts of '
    'the input sequence. Transformers replaced RNNs and LSTMs as the dominant architecture '
    'for NLP tasks. BERT, GPT, and T5 are all based on the Transformer architecture.'
)

questions = [
    'When was the Transformer architecture introduced?',
    'What was the key innovation?',
    'What did Transformers replace?',
    'Which models are based on Transformers?',
]

results: list[dict[str, str]] = []
for question in questions:
    answer = qa_pipeline(question=question, context=context)
    results.append({
        'Question': question,
        'Answer': answer['answer'],
        'Score': f'{answer["score"]:.4f}',
        'Position': f'{answer["start"]}-{answer["end"]}',
    })

pd.DataFrame(results)

Notice how the model extracts exact spans from the context — it never generates new text. The confidence score tells us how certain the model is about each answer. Scores above 0.7 are generally reliable; below 0.3 suggests the answer may not be in the context.

## Method 2: Manual Model Loading

For deeper understanding, let's load the model manually and examine the raw start/end logits that the model produces. This reveals exactly how the model selects answer spans.

In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(device)
model.eval()

print(f'Model: {MODEL_NAME}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
def extract_answer(
    question: str,
    context: str,
    tokenizer: transformers.PreTrainedTokenizer,
    model: torch.nn.Module,
    top_k: int = 3,
) -> list[dict[str, object]]:
    """Extract top-k answer candidates from a context.

    Args:
        question: The question to answer.
        context: The context passage containing the answer.
        tokenizer: The QA model's tokenizer.
        model: The QA model.
        top_k: Number of answer candidates to return.

    Returns:
        List of answer dictionaries with text, score, start, and end.
    """
    inputs = tokenizer(
        question, context,
        return_tensors='pt',
        return_offsets_mapping=True,
        truncation=True,
        max_length=512,
    )
    offset_mapping = inputs.pop('offset_mapping')[0]
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]
    
    # Convert to probabilities
    start_probs = torch.softmax(start_logits, dim=-1)
    end_probs = torch.softmax(end_logits, dim=-1)
    
    # Find the context token range (skip question tokens)
    # The tokenizer encodes [CLS] question [SEP] context [SEP]
    input_ids = inputs['input_ids'][0]
    sep_indices = (input_ids == tokenizer.sep_token_id).nonzero(as_tuple=True)[0]
    context_start = sep_indices[0].item() + 1 if len(sep_indices) > 0 else 0
    context_end = len(input_ids) - 1
    
    # Find top-k start and end positions in the context
    start_top = torch.topk(start_probs[context_start:context_end], min(top_k * 2, context_end - context_start))
    end_top = torch.topk(end_probs[context_start:context_end], min(top_k * 2, context_end - context_start))
    
    # Generate answer candidates
    candidates: list[dict[str, object]] = []
    for s_idx, s_score in zip(start_top.indices, start_top.values):
        for e_idx, e_score in zip(end_top.indices, end_top.values):
            s = s_idx.item() + context_start
            e = e_idx.item() + context_start
            if e >= s and (e - s) < 30:  # Valid span, not too long
                start_char = offset_mapping[s][0].item()
                end_char = offset_mapping[e][1].item()
                candidates.append({
                    'answer': context[start_char:end_char],
                    'score': (s_score.item() * e_score.item()),
                    'start': start_char,
                    'end': end_char,
                })
    
    # Sort by score and return top-k
    candidates.sort(key=lambda x: x['score'], reverse=True)
    return candidates[:top_k]


# Demo: Get top-3 answer candidates
question = 'What was the key innovation of Transformers?'
candidates = extract_answer(question, context, tokenizer, model, top_k=3)

print(f'Question: "{question}"\n')
print('Top-3 answer candidates:')
pd.DataFrame([{
    'Rank': i + 1,
    'Answer': c['answer'],
    'Score': f'{c["score"]:.6f}',
    'Span': f'{c["start"]}-{c["end"]}',
} for i, c in enumerate(candidates)])

The manual approach reveals that the model actually produces two probability distributions: one for the start position and one for the end position. By examining the top-k candidates from each, we can generate multiple answer hypotheses and rank them. This is useful in production systems where you want to present multiple possible answers.

## Practical Applications

Extractive QA is widely used in customer support, documentation search, and knowledge base systems. Let's explore some practical scenarios.

In [ ]:
# Application 1: Multi-question document analysis
def document_qa(
    context: str,
    questions: list[str],
    qa_pipe: transformers.Pipeline,
) -> pd.DataFrame:
    """Answer multiple questions about a document.

    Args:
        context: The document text.
        questions: List of questions to answer.
        qa_pipe: HuggingFace QA pipeline.

    Returns:
        DataFrame with questions, answers, and confidence scores.
    """
    results: list[dict[str, str]] = []
    for question in questions:
        answer = qa_pipe(question=question, context=context)
        confidence = 'High' if answer['score'] > 0.7 else 'Medium' if answer['score'] > 0.3 else 'Low'
        results.append({
            'Question': question,
            'Answer': answer['answer'],
            'Score': f'{answer["score"]:.4f}',
            'Confidence': confidence,
        })
    return pd.DataFrame(results)


# Product documentation scenario
product_doc = (
    'The XR-500 wireless headphones feature active noise cancellation with '
    'three adjustable levels. Battery life is up to 30 hours with ANC off, '
    'or 20 hours with ANC on. They connect via Bluetooth 5.3 and support '
    'multipoint connection to two devices simultaneously. The headphones '
    'weigh 250 grams and are available in black, white, and midnight blue. '
    'The price starts at $199 for the standard edition and $249 for the '
    'premium edition with carrying case. Warranty is 2 years.'
)

product_questions = [
    'How long does the battery last?',
    'What Bluetooth version do they use?',
    'How much do the headphones weigh?',
    'What colors are available?',
    'What is the warranty period?',
    'What is the price?',
]

print('=== Product Documentation QA ===')
document_qa(product_doc, product_questions, qa_pipeline)

In [ ]:
# Application 2: Wikipedia-style knowledge extraction
wiki_context = (
    'Python is a high-level, general-purpose programming language. Its design '
    'philosophy emphasizes code readability with the use of significant '
    'indentation. Python is dynamically typed and garbage-collected. It '
    'supports multiple programming paradigms, including structured, '
    'object-oriented and functional programming. Guido van Rossum began '
    'working on Python in the late 1980s as a successor to the ABC '
    'programming language and first released it in 1991 as Python 0.9.0. '
    'Python consistently ranks as one of the most popular programming '
    'languages.'
)

wiki_questions = [
    'Who created Python?',
    'When was Python first released?',
    'What does Python emphasize?',
    'What programming paradigms does Python support?',
    'What language was Python a successor to?',
]

print('=== Wikipedia Knowledge Extraction ===')
document_qa(wiki_context, wiki_questions, qa_pipeline)

## Handling Edge Cases

Real-world QA systems need to handle tricky situations: unanswerable questions, long documents, and ambiguous queries. Let's explore these challenges.

In [ ]:
# Edge case 1: Question not answerable from context
unanswerable_context = 'The Eiffel Tower is located in Paris, France. It was built in 1889.'
unanswerable_questions = [
    'Where is the Eiffel Tower?',           # Answerable
    'How tall is the Eiffel Tower?',         # NOT in context
    'Who designed the Eiffel Tower?',        # NOT in context
]

print('=== Unanswerable Question Detection ===')
print(f'Context: "{unanswerable_context}"\n')

edge_results: list[dict[str, str]] = []
for q in unanswerable_questions:
    answer = qa_pipeline(question=q, context=unanswerable_context)
    answerable = 'Yes' if answer['score'] > 0.5 else 'Probably Not'
    edge_results.append({
        'Question': q,
        'Answer': answer['answer'],
        'Score': f'{answer["score"]:.4f}',
        'Answerable?': answerable,
    })

pd.DataFrame(edge_results)

Notice how the model still produces an answer even for unanswerable questions — it just has lower confidence. In production, you should set a threshold (e.g., 0.3) below which you respond with "I don't have enough information to answer that." SQuAD v2 models (like `deepset/roberta-base-squad2`) are specifically trained to detect unanswerable questions.

## Performance Benchmarking

Let's measure QA inference speed across different context lengths.

In [ ]:
def benchmark_qa(
    contexts: list[str],
    question: str,
    qa_pipe: transformers.Pipeline,
    num_runs: int = 5,
) -> pd.DataFrame:
    """Benchmark QA pipeline across different context lengths.

    Args:
        contexts: List of contexts of varying lengths.
        question: A question applicable to all contexts.
        qa_pipe: HuggingFace QA pipeline.
        num_runs: Number of runs for averaging.

    Returns:
        DataFrame with timing results per context length.
    """
    results: list[dict[str, str]] = []
    
    for ctx in contexts:
        word_count = len(ctx.split())
        
        # Warmup
        qa_pipe(question=question, context=ctx)
        
        times: list[float] = []
        for _ in range(num_runs):
            start = time.perf_counter()
            qa_pipe(question=question, context=ctx)
            times.append(time.perf_counter() - start)
        
        results.append({
            'Context Words': str(word_count),
            'Mean Latency (ms)': f'{np.mean(times) * 1000:.1f}',
            'Std (ms)': f'{np.std(times) * 1000:.1f}',
            'Device': str(device),
        })
    
    return pd.DataFrame(results)


# Create contexts of different lengths
base_sentence = 'The Transformer architecture was introduced in 2017. '
short_context = base_sentence * 3
medium_context = base_sentence * 15
long_context = base_sentence * 50

print('=== QA Latency vs Context Length ===')
benchmark_qa(
    [short_context, medium_context, long_context],
    'When was the Transformer introduced?',
    qa_pipeline,
)

## Exercises

1. **FAQ bot**: Build a simple FAQ system — given a list of FAQ entries (question + answer paragraph), use the QA model to find answers to user queries.

2. **SQuAD v2**: Load `deepset/roberta-base-squad2` and test it on unanswerable questions. Compare its behavior with the DistilBERT model we used.

3. **Confidence calibration**: Run 20 questions on a Wikipedia paragraph. Plot the relationship between confidence score and answer correctness. Is the model well-calibrated?

4. **Multi-hop QA**: Try asking questions that require combining information from multiple sentences. How well does the model handle these?

## Key Takeaways

- **Extractive QA** finds answer spans within a context — it does not generate new text
- The model predicts **start and end positions** independently, then combines them into answer spans
- **Confidence scores** are critical for production use — set thresholds to avoid returning unreliable answers
- **SQuAD v1 models** always return an answer; **SQuAD v2 models** can detect unanswerable questions
- QA latency increases with context length — for large documents, consider chunking or retrieval (RAG)

## Next Steps & Resources

**Next notebook**: [01_06 — Translation & Multilingual](01_06_nlp_translation.ipynb) — machine translation and multilingual models.

**Resources:**
- [HuggingFace QA Tutorial](https://huggingface.co/docs/transformers/tasks/question_answering)
- [SQuAD Dataset](https://rajpurkar.github.io/SQuAD-explorer/) — Standard QA benchmark
- [QA Models on Hub](https://huggingface.co/models?pipeline_tag=question-answering)
- [SQuAD v2 Paper](https://arxiv.org/abs/1806.03822) — Know What You Don't Know